# Sinha Rotor — Natural Frequency Analysis

Build the Sinha rotor model using ROSS and extract its natural frequencies.

**Physical dimensions (Sinha's paper):**
| Parameter | Value |
|---|---|
| Shaft OD | 10 mm |
| Shaft ID | 0 (solid) |
| Shaft length | 550 mm |
| Material | Steel (ρ = 7810 kg/m³, E = 211 GPa, G = 81.2 GPa) |
| Bush bearings | at 20 mm and 510 mm |
| Balance disk OD | 75 mm |
| Balance disk ID | 10 mm |
| Disk thickness | 25 mm |
| Disk location | 275 mm (midspan) |
| Target Fn1 | ≈ 27.50 Hz |

In [2]:
import numpy as np
import ross as rs
from ross.materials import Material

print(f"ROSS version: {rs.__version__}")

ROSS version: 2.0.0


## 1. Define Material and Shaft Geometry

In [3]:
# --- Material ---
steel = Material(name="Steel_Sinha", rho=7810, E=211e9, G_s=81.2e9)

# --- Shaft geometry ---
shaft_od = 0.010   # 10 mm outer diameter
shaft_id = 0.0     # solid shaft

# Node positions along the shaft [mm]
#   key locations: 0 — bearing1(20) — ... — disk(275) — ... — bearing2(510) — 550
node_pos_mm = [0, 20, 70, 120, 170, 220, 275, 325, 375, 425, 475, 510, 550]

# Element lengths [m]
lengths_m = [
    (node_pos_mm[i + 1] - node_pos_mm[i]) * 1e-3
    for i in range(len(node_pos_mm) - 1)
]

print(f"Number of elements : {len(lengths_m)}")
print(f"Number of nodes    : {len(node_pos_mm)}")
print(f"Element lengths [mm]: {[L*1e3 for L in lengths_m]}")

Number of elements : 12
Number of nodes    : 13
Element lengths [mm]: [20.0, 50.0, 50.0, 50.0, 50.0, 55.0, 50.0, 50.0, 50.0, 50.0, 35.0, 40.0]


## 2. Create Shaft Elements

In [5]:
shaft_elements = [
    rs.ShaftElement(
        L=le,
        idl=shaft_id,
        odl=shaft_od,
        material=steel,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for le in lengths_m
]

print(f"Created {len(shaft_elements)} shaft elements")

Created 12 shaft elements


## 3. Create Disk Element

Balance disk at node 6 (275 mm ≈ midspan): OD 75 mm, ID 10 mm, thickness 25 mm.

In [6]:
disk = rs.DiskElement.from_geometry(
    n=6,
    material=steel,
    width=0.025,    # 25 mm thick
    i_d=0.010,      # ID = 10 mm (matches shaft OD)
    o_d=0.075,      # OD = 75 mm
)

print(f"Disk at node 6 (275 mm)")
print(f"  Mass : {disk.m:.4f} kg")
print(f"  Ip   : {disk.Ip:.6e} kg·m²")
print(f"  Id   : {disk.Id:.6e} kg·m²")

Disk at node 6 (275 mm)
  Mass : 0.8473 kg
  Ip   : 6.063156e-04 kg·m²
  Id   : 3.472855e-04 kg·m²


## 4. Create Bearing Elements and Tune Stiffness

Bush bearings at node 1 (20 mm) and node 11 (510 mm). The bearing stiffness is iteratively tuned so that the first natural frequency matches the target Fn1 ≈ 27.5 Hz.

In [ ]:
target_fn1 = 27.5       # Hz
target_wn1 = 2.0 * np.pi * target_fn1
k_bearing  = 1.0e5      # initial guess [N/m]
c_bearing  = 10.0        # light damping [N·s/m]

print("Iterative bearing-stiffness tuning:")
print(f"  Target Fn1 = {target_fn1} Hz\n")

for iteration in range(15):
    brg1 = rs.BearingElement(n=1,  kxx=k_bearing, kyy=k_bearing,
                              cxx=c_bearing, cyy=c_bearing)
    brg2 = rs.BearingElement(n=11, kxx=k_bearing, kyy=k_bearing,
                              cxx=c_bearing, cyy=c_bearing)

    rotor = rs.Rotor(shaft_elements, [disk], [brg1, brg2], tag="Sinha Rotor")
    modal = rotor.run_modal(speed=0)
    wn1 = modal.wn[0]
    fn1 = wn1 / (2.0 * np.pi)

    print(f"  iter {iteration+1:2d} : k_brg = {k_bearing:12.1f} N/m  →  Fn1 = {fn1:.4f} Hz")

    if abs(fn1 - target_fn1) / target_fn1 < 0.005:
        print(f"\n  ✓ Converged! Fn1 = {fn1:.2f} Hz  (error < 0.5%)")
        break

    k_bearing *= (target_wn1 / wn1) ** 2

print(f"\n  Final k_bearing = {k_bearing:.1f} N/m")

Iterative bearing-stiffness tuning:
  Target Fn1 = 27.5 Hz



## 5. Rotor Summary and Geometry

In [9]:
print("Sinha Rotor Model Summary")
print("=" * 40)
print(f"  Tag            : {rotor.tag}")
print(f"  Nodes          : {len(rotor.nodes)}")
print(f"  DOFs           : {rotor.ndof}")
print(f"  Shaft elements : {len(rotor.shaft_elements)}")
print(f"  Disk elements  : {len(rotor.disk_elements)}")
print(f"  Bearing elms   : {len(rotor.bearing_elements)}")

rotor.plot_rotor()

Sinha Rotor Model Summary
  Tag            : Sinha Rotor
  Nodes          : 13
  DOFs           : 78
  Shaft elements : 12
  Disk elements  : 1
  Bearing elms   : 2


## 6. Natural Frequencies (Modal Analysis at 0 RPM)

Extract and display the first natural frequencies of the Sinha rotor at standstill (0 RPM).

In [10]:
# Modal analysis at 0 RPM
modal = rotor.run_modal(speed=0)

n_modes = min(10, len(modal.wn))

print("Natural Frequencies at 0 RPM")
print("=" * 50)
print(f"{'Mode':>5s}  {'ωn [rad/s]':>12s}  {'fn [Hz]':>10s}  {'ζ (damping ratio)':>18s}")
print("-" * 50)

for i in range(n_modes):
    wn_i = modal.wn[i]
    fn_i = wn_i / (2.0 * np.pi)
    # damping ratio from eigenvalues
    evalues = modal.evalues
    # paired eigenvalues: take the one with positive imaginary part
    zeta_i = -np.real(evalues[2*i]) / np.abs(evalues[2*i]) if 2*i < len(evalues) else 0.0
    print(f"{i+1:>5d}  {wn_i:>12.3f}  {fn_i:>10.3f}  {zeta_i:>18.6f}")

print(f"\n→ First natural frequency: Fn1 = {modal.wn[0]/(2*np.pi):.2f} Hz")

Natural Frequencies at 0 RPM
 Mode    ωn [rad/s]     fn [Hz]   ζ (damping ratio)
--------------------------------------------------
    1       185.875      29.583            0.001784
    2       185.875      29.583            0.042316
    3      1059.140     168.567            0.053331
    4      1059.140     168.567            0.023069
    5      1435.311     228.437            0.009377
    6      1435.311     228.437            0.009377

→ First natural frequency: Fn1 = 29.58 Hz


## 7. Campbell Diagram

Visualise how the natural frequencies change with rotor speed (gyroscopic effects).

In [ ]:
# Campbell diagram — speed range 0 to 5000 RPM
speed_range = np.linspace(0, 5000 * 2 * np.pi / 60, 50)  # rad/s
camp = rotor.run_campbell(speed_range)
fig = camp.plot()
fig.show()

## 8. Mode Shapes

Plot the first few mode shapes at 0 RPM to visualise the deformation patterns.

In [ ]:
# Plot first 4 mode shapes
modal_0 = rotor.run_modal(speed=0)
fig = modal_0.plot_mode_3d(mode=0)
fig.show()

In [ ]:
fig = modal_0.plot_mode_3d(mode=1)
fig.show()

## 9. Natural Frequencies at Operating Speed (1500 RPM)

Check how gyroscopic effects shift the natural frequencies at the operating speed of 1500 RPM (25 Hz).

In [ ]:
speed_1500 = 1500 * 2 * np.pi / 60  # rad/s
modal_1500 = rotor.run_modal(speed=speed_1500)

n_modes = min(10, len(modal_1500.wn))

print(f"Natural Frequencies at 1500 RPM ({speed_1500:.2f} rad/s)")
print("=" * 50)
print(f"{'Mode':>5s}  {'ωn [rad/s]':>12s}  {'fn [Hz]':>10s}")
print("-" * 50)

for i in range(n_modes):
    wn_i = modal_1500.wn[i]
    fn_i = wn_i / (2.0 * np.pi)
    print(f"{i+1:>5d}  {wn_i:>12.3f}  {fn_i:>10.3f}")

print(f"\n→ Fn1 at 1500 RPM = {modal_1500.wn[0]/(2*np.pi):.2f} Hz")
print(f"→ Fn1 at   0 RPM = {modal_0.wn[0]/(2*np.pi):.2f} Hz")